In [ ]:
!pip install tensorflow scikit-learn joblib


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,
    mean_absolute_percentage_error,
    classification_report,
    f1_score,
    explained_variance_score,
    max_error,
    confusion_matrix
)


**LOAD & PREP DATA**

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving combined_final_with_lags.csv to combined_final_with_lags.csv


In [ ]:
df = pd.read_csv("combined_final_with_lags.csv")

df = df.sort_values(
    by=['City_encoded', 'Year', 'Month']
).reset_index(drop=True)


**PAST-ONLY FEATURES (NO LEAKAGE)**

In [ ]:
df['AQI_roll3_past'] = df.groupby('City_encoded')['avg_AQI'].shift(1).rolling(3).mean()
df['AQI_roll6_past'] = df.groupby('City_encoded')['avg_AQI'].shift(1).rolling(6).mean()

df['Day_roll3_past'] = df.groupby('City_encoded')['Day'].shift(1).rolling(3).mean()
df['Night_roll3_past'] = df.groupby('City_encoded')['Night'].shift(1).rolling(3).mean()

df = df.dropna().reset_index(drop=True)


**CYCLICAL TIME**

In [ ]:
df['month_sin'] = np.sin(2*np.pi*df['Month']/12)
df['month_cos'] = np.cos(2*np.pi*df['Month']/12)


**NOISE BUCKETS**

In [ ]:
def day_risk(x):
    if x < 55: return 0
    elif x < 65: return 1
    else: return 2

def night_risk(x):
    if x < 45: return 0
    elif x < 55: return 1
    else: return 2

df['Day_risk'] = df['Day'].apply(day_risk)
df['Night_risk'] = df['Night'].apply(night_risk)


**FEATURE SET**

In [ ]:
FEATURES = [
    'avg_AQI_lag1','avg_AQI_lag2','avg_AQI_lag3',
    'AQI_roll3_past','AQI_roll6_past',
    'Day_lag1','Day_lag2','Day_roll3_past',
    'Night_lag1','Night_lag2','Night_roll3_past',
    'month_sin','month_cos'
]


**SEQUENCE BUILDER (PER CITY)**

In [ ]:
def build_sequences(df, window=6):
    X, y_aqi, y_day, y_night = [], [], [], []

    for city in df['City_encoded'].unique():
        cdf = df[df['City_encoded']==city].sort_values(['Year','Month'])
        vals = cdf[FEATURES + ['avg_AQI','Day_risk','Night_risk']].values
        f = len(FEATURES)

        for i in range(window, len(vals)):
            X.append(vals[i-window:i,:f])
            y_aqi.append(vals[i,f])
            y_day.append(vals[i,f+1])
            y_night.append(vals[i,f+2])

    return np.array(X), np.array(y_aqi), np.array(y_day), np.array(y_night)


In [ ]:
X, y_aqi, y_day, y_night = build_sequences(df)


**TRAIN–TEST SPLIT & SCALING**

In [ ]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_aqi_train, y_aqi_test = y_aqi[:split], y_aqi[split:]
y_day_train, y_day_test = y_day[:split], y_day[split:]
y_night_train, y_night_test = y_night[:split], y_night[split:]


In [ ]:
scaler = StandardScaler()
n_feat = X_train.shape[2]

X_train = scaler.fit_transform(
    X_train.reshape(-1,n_feat)
).reshape(X_train.shape)

X_test = scaler.transform(
    X_test.reshape(-1,n_feat)
).reshape(X_test.shape)


**TRANSFORMER BLOCKS**

In [ ]:
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, Conv1D, Flatten
)
from tensorflow.keras.models import Model


**Vanilla Transformer**

In [ ]:
def vanilla_transformer(x, heads, key_dim, ff_dim, dropout, activation):
    attn = MultiHeadAttention(num_heads=heads, key_dim=key_dim)(x,x)
    x = LayerNormalization()(x+attn)
    ff = Dense(ff_dim, activation=activation)(x)
    ff = Dense(x.shape[-1])(ff)
    return LayerNormalization()(x+ff)


**Hybrid Conv + Attention**

In [ ]:
def conv_attn_transformer(x, heads, key_dim, ff_dim, dropout, activation):
    x = Conv1D(32,3,padding='same',activation=activation)(x)
    attn = MultiHeadAttention(num_heads=heads, key_dim=key_dim)(x,x)
    x = LayerNormalization()(x+attn)
    ff = Dense(ff_dim, activation=activation)(x)
    ff = Dense(x.shape[-1])(ff)
    return LayerNormalization()(x+ff)


**Simplified TFT**

In [ ]:
def tft_block(x, heads, key_dim, ff_dim, dropout, activation):
    attn = MultiHeadAttention(num_heads=heads, key_dim=key_dim)(x,x)
    x = LayerNormalization()(x+attn)
    gated = Dense(ff_dim, activation=activation)(x)
    gated = Dense(x.shape[-1], activation='sigmoid')(gated)
    return LayerNormalization()(x+gated)


**MODEL BUILDER**

In [ ]:
def build_model(block_fn, block_cfg, train_cfg):
    inp = Input(shape=X_train.shape[1:])

    x = block_fn(inp, **block_cfg)
    x = Dropout(block_cfg['dropout'])(x)
    x = Flatten()(x)

    shared = Dense(128, activation=block_cfg['activation'])(x)

    # AQI regression
    aqi_out = Dense(1, name='aqi')(shared)

    # Noise classification
    noise_hidden = Dense(64, activation=block_cfg['activation'])(shared)
    day_out = Dense(3, activation='softmax', name='day')(noise_hidden)
    night_out = Dense(3, activation='softmax', name='night')(noise_hidden)

    model = Model(inp, [aqi_out, day_out, night_out])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(train_cfg['lr']),
        loss={
            'aqi': 'mse',
            'day': 'sparse_categorical_crossentropy',
            'night': 'sparse_categorical_crossentropy'
        },
        loss_weights={'aqi': 1.0, 'day': 0.6, 'night': 0.6}
    )

    return model


**TRAIN & EVALUATE FUNCTION**

In [ ]:
def train_and_eval(name, block_fn, block_cfg, train_cfg):
    model = build_model(block_fn, block_cfg, train_cfg)

    model.fit(
        X_train,
        {
            'aqi': y_aqi_train,
            'day': y_day_train,
            'night': y_night_train
        },
        epochs=80,
        batch_size=32,
        validation_split=0.1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                patience=12,
                restore_best_weights=True
            )
        ],
        verbose=0
    )

    aqi_p, day_p, night_p = model.predict(X_test, verbose=0)

    return {
        'Model': name,
        'AQI_R2': r2_score(y_aqi_test, aqi_p),
        'AQI_MAPE': mean_absolute_percentage_error(y_aqi_test, aqi_p) * 100,
        'Day_High_F1': f1_score(
            y_day_test,
            np.argmax(day_p, axis=1),
            labels=[2],
            average='macro'
        ),
        'Night_High_F1': f1_score(
            y_night_test,
            np.argmax(night_p, axis=1),
            labels=[2],
            average='macro'
        )
    }


**EXPERIMENT CONFIGS**

In [ ]:
configs = [
    {
        'name': 'Vanilla-GELU',
        'block': vanilla_transformer,
        'block_cfg': {
            'heads': 4,
            'key_dim': 32,
            'ff_dim': 128,
            'dropout': 0.2,
            'activation': 'gelu'
        },
        'train_cfg': {
            'lr': 5e-4
        }
    },
    {
        'name': 'Conv-Attn-GELU',
        'block': conv_attn_transformer,
        'block_cfg': {
            'heads': 4,
            'key_dim': 32,
            'ff_dim': 128,
            'dropout': 0.3,
            'activation': 'gelu'
        },
        'train_cfg': {
            'lr': 5e-4
        }
    },
    {
        'name': 'TFT-Simplified',
        'block': tft_block,
        'block_cfg': {
            'heads': 4,
            'key_dim': 32,
            'ff_dim': 128,
            'dropout': 0.3,
            'activation': 'gelu'
        },
        'train_cfg': {
            'lr': 3e-4
        }
    }
]


**RUN ALL EXPERIMENTS**

In [ ]:
results = []

for exp in configs:
    res = train_and_eval(
        exp['name'],
        exp['block'],
        exp['block_cfg'],
        exp['train_cfg']
    )
    results.append(res)
    print(res)


{'Model': 'Vanilla-GELU', 'AQI_R2': 0.9242840524828548, 'AQI_MAPE': 7.4073045893873575, 'Day_High_F1': 0.72, 'Night_High_F1': 0.9168704156479217}
{'Model': 'Conv-Attn-GELU', 'AQI_R2': 0.8680795805843606, 'AQI_MAPE': 9.831592911922145, 'Day_High_F1': 0.7115097159940209, 'Night_High_F1': 0.9061728395061729}
{'Model': 'TFT-Simplified', 'AQI_R2': 0.9049937598900243, 'AQI_MAPE': 8.885059633433546, 'Day_High_F1': 0.7280575539568346, 'Night_High_F1': 0.9168704156479217}


**FINAL RESULTS TABLE**

In [ ]:
pd.DataFrame(results)


,Model,AQI_R2,AQI_MAPE,Day_High_F1,Night_High_F1
0,Vanilla-GELU,0.925684,6.587907,0.684848,0.898876
1,Conv-Attn-GELU,0.871220,11.015557,0.737892,0.915545
2,TFT-Simplified,0.885105,9.825262,0.557621,0.894737
